In [2]:
import os
import sys

# Đảm bảo Notebook luôn đứng ở thư mục gốc dự án
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"✅ Đang chạy tại thư mục: {os.getcwd()}")

✅ Đang chạy tại thư mục: d:\HCMUT\NMTTNT\ai-ambulance-coordinator


In [5]:
from modules.core.disease_classifier import DiseaseClassifier

print("=== CHẠY BẢN SCALE-UP: SKLEARN NAIVE BAYES ===")
clf = DiseaseClassifier(model_name="sklearnNaiveBayes")

# Truyền thẳng đường dẫn file CSV vào đây (Đảm bảo file dataset.csv đang nằm trong thư mục data/)
clf.train("data/Final_Augmented_dataset_Diseases_and_Symptoms.csv") 

# In bảng đánh giá
print("\n--- BẢNG ĐIỂM ---")
metrics = clf.evaluate()
for m, score in metrics.items():
    print(f"{m:<10}: {score * 100:.2f}%")

# Test thử chẩn đoán (Lấy tự động 2 triệu chứng đầu tiên có thật trong data)
print("\n--- TEST DỰ ĐOÁN ---")
# Trích xuất chính xác tên 2 triệu chứng từ từ điển của AI
symptom_1 = clf.symptoms_vocab[0]
symptom_2 = clf.symptoms_vocab[5] # Lấy triệu chứng số 1 và số 6

symptoms = [symptom_1, symptom_2]
print(f"Triệu chứng đầu vào: {symptoms}")
print(f"=> AI Chẩn đoán: {clf.predict(symptoms)}")

=== CHẠY BẢN SCALE-UP: SKLEARN NAIVE BAYES ===
Đang tải dữ liệu từ data/Final_Augmented_dataset_Diseases_and_Symptoms.csv...
Đang chuyển đổi sang Sparse Matrix để tăng tốc...
Đang huấn luyện mô hình sklearnNaiveBayes...
[HOÀN TẤT] Huấn luyện thành công!

--- BẢNG ĐIỂM ---
Đang đánh giá mô hình trên tập Test...
Accuracy  : 84.53%
Precision : 84.32%
Recall    : 84.53%
F1-Score  : 83.87%

--- TEST DỰ ĐOÁN ---
Triệu chứng đầu vào: ['anxiety and nervousness', 'dizziness']
=> AI Chẩn đoán: acute stress reaction


In [6]:
from modules.core.disease_classifier import DiseaseClassifier

print("=== CHẠY BẢN LÝ THUYẾT: NAIVE BAYES TỪ SCRATCH (NUMPY) ===")
# 1. Gọi tên mô hình mà bạn đã dày công code bằng tay!
clf_simple = DiseaseClassifier(model_name="simpleClassifierModel")

# 2. Huấn luyện trên cùng một bộ dữ liệu
# Lưu ý: Code thuần chạy có thể sẽ mất thêm vài giây so với bản C++ tối ưu của thư viện
clf_simple.train("data/Final_Augmented_dataset_Diseases_and_Symptoms.csv") 

# 3. In bảng đánh giá để so sánh điểm
print("\n--- BẢNG ĐIỂM ---")
metrics_simple = clf_simple.evaluate()
for m, score in metrics_simple.items():
    print(f"{m:<10}: {score * 100:.2f}%")

# 4. Test thử chẩn đoán với đúng 2 triệu chứng giống hệt cell ở trên
print("\n--- TEST DỰ ĐOÁN ---")
symptom_1 = clf_simple.symptoms_vocab[0]
symptom_2 = clf_simple.symptoms_vocab[5] 

symptoms = [symptom_1, symptom_2]
print(f"Triệu chứng đầu vào: {symptoms}")
print(f"=> AI Chẩn đoán: {clf_simple.predict(symptoms)}")

=== CHẠY BẢN LÝ THUYẾT: NAIVE BAYES TỪ SCRATCH (NUMPY) ===
Đang tải dữ liệu từ data/Final_Augmented_dataset_Diseases_and_Symptoms.csv...
Đang chuyển đổi sang Sparse Matrix để tăng tốc...
Đang huấn luyện mô hình simpleClassifierModel...
[HOÀN TẤT] Huấn luyện thành công!

--- BẢNG ĐIỂM ---
Đang đánh giá mô hình trên tập Test...
Accuracy  : 84.53%
Precision : 84.32%
Recall    : 84.53%
F1-Score  : 83.87%

--- TEST DỰ ĐOÁN ---
Triệu chứng đầu vào: ['anxiety and nervousness', 'dizziness']
=> AI Chẩn đoán: acute stress reaction


In [3]:
pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 6.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import difflib # Thư viện "thần thánh" để dò lỗi chính tả

# --- 1. TẠO CÁC THÀNH PHẦN GIAO DIỆN ---
all_symptoms = list(clf.symptoms_vocab)

# Khung 1: Nhập liệu bằng chữ (Hỗ trợ gõ sai chính tả)
text_input = widgets.Text(
    value='',
    placeholder='VD: headache, stomach pain, fever...',
    description='Gõ nhanh:',
    disabled=False,
    layout=widgets.Layout(width='80%')
)

# Khung 2: Chọn thủ công (Dành cho ai thích click chuột)
symptom_selector = widgets.SelectMultiple(
    options=all_symptoms,
    description='Hoặc chọn:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='120px')
)

# Nút bấm Chẩn đoán
predict_btn = widgets.Button(
    description='🚀 Chẩn Đoán Ngay',
    button_style='success',
    tooltip='Bấm để xem AI chẩn đoán',
    icon='stethoscope'
)

# Khu vực hiển thị kết quả
output_area = widgets.Output()

# --- 2. HÀM XỬ LÝ KHI BẤM NÚT ---
def on_predict_button_clicked(b):
    with output_area:
        clear_output()
        
        # --- BƯỚC A: XỬ LÝ PHẦN TEXT TỰ GÕ (Fuzzy Matching) ---
        raw_text = text_input.value
        detected_symptoms = []
        unrecognized = []
        
        if raw_text.strip():
            # Tách các từ bằng dấu phẩy
            input_list = [s.strip() for s in raw_text.split(',')]
            for word in input_list:
                if word:
                    # Dò tìm từ giống nhất với độ chính xác >= 60%
                    matches = difflib.get_close_matches(word, all_symptoms, n=1, cutoff=0.6)
                    if matches:
                        detected_symptoms.append(matches[0])
                    else:
                        unrecognized.append(word)
        
        # --- BƯỚC B: LẤY THÊM TỪ KHUNG CHỌN CHUỘT ---
        selected_from_list = list(symptom_selector.value)
        
        # --- BƯỚC C: GỘP CHUNG VÀ CHUẨN BỊ CHẨN ĐOÁN ---
        # Dùng set() để lỡ người dùng vừa gõ vừa chọn thì không bị trùng lặp
        final_symptoms = list(set(detected_symptoms + selected_from_list))
        
        # Báo lỗi nếu không có dữ liệu
        if not final_symptoms:
            print("⚠️ CẢNH BÁO: Vui lòng nhập hoặc chọn ít nhất 1 triệu chứng!")
            return
            
        # Thông báo nếu có từ gõ sai quá mức AI không đoán được
        if unrecognized:
            print(f"⚠️ Cảnh báo: AI không hiểu các từ '{unrecognized}'. Đã tự động bỏ qua!")
            
        print(f"✅ Đã nhận diện triệu chứng: {final_symptoms}")
        print("=" * 60)
        
        # --- CHẠY MÔ HÌNH 1 (SKLEARN) ---
        start1 = time.time()
        pred1 = clf.predict(final_symptoms)
        time1 = time.time() - start1
        
        print("🧠 MÔ HÌNH 1: SCALE-UP (SCIKIT-LEARN)")
        print(f"   => 🩺 Bệnh dự đoán: {pred1.upper()}")
        print(f"   => ⏱️ Thời gian xử lý: {time1:.5f} giây\n")
        
        # --- CHẠY MÔ HÌNH 2 (TỰ CODE) ---
        start2 = time.time()
        pred2 = clf_simple.predict(final_symptoms)
        time2 = time.time() - start2
        
        print("🛠️ MÔ HÌNH 2: FROM SCRATCH (NUMPY MATH)")
        print(f"   => 🩺 Bệnh dự đoán: {pred2.upper()}")
        print(f"   => ⏱️ Thời gian xử lý: {time2:.5f} giây\n")
        
        # --- ĐỐI CHIẾU ---
        print("-" * 60)
        if pred1 == pred2:
            print("🎉 KẾT LUẬN: Xuất sắc! 2 mô hình đồng thuận 100% kết quả.")
        else:
            print("⚠️ KẾT LUẬN: Có sự sai lệch giữa 2 mô hình.")

# Gắn sự kiện click cho nút
predict_btn.on_click(on_predict_button_clicked)

# --- 3. HIỂN THỊ GIAO DIỆN RA MÀN HÌNH ---
display(widgets.HTML("<h2 style='color:#2E86C1;'>🚑 Giao Diện AI Nhập Liệu Thông Minh</h2>"))
display(widgets.HTML("<i><b>Hướng dẫn:</b> Bạn có thể gõ nhanh triệu chứng vào ô <b>Gõ nhanh</b> (cách nhau bằng dấu phẩy, cố tình gõ sai chính tả để test AI). Hoặc dùng chuột chọn từ hộp bên dưới.</i><br><br>"))

display(text_input)
display(symptom_selector)
display(predict_btn)
display(output_area)

HTML(value="<h2 style='color:#2E86C1;'>🚑 Giao Diện AI Nhập Liệu Thông Minh</h2>")

HTML(value='<i><b>Hướng dẫn:</b> Bạn có thể gõ nhanh triệu chứng vào ô <b>Gõ nhanh</b> (cách nhau bằng dấu phẩ…

Text(value='', description='Gõ nhanh:', layout=Layout(width='80%'), placeholder='VD: headache, stomach pain, f…

SelectMultiple(description='Hoặc chọn:', layout=Layout(height='120px', width='80%'), options=('anxiety and ner…

Button(button_style='success', description='🚀 Chẩn Đoán Ngay', icon='stethoscope', style=ButtonStyle(), toolti…

Output()

In [6]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)